# 다봐요 (Dabwayo) — Minimal GPU t2v server (free Colab)

Ultra-light text→video server for the **free T4**. One small model
(ModelScope `text-to-video-ms-1.7b` — no giant T5 encoder, the usual
free-Colab OOM cause is gone). Serves Dabwayo's `remote` provider.

## ⚠️ Before running
**Runtime → Change runtime type → Hardware accelerator = GPU (T4) → Save.**
Then **Runtime → Restart and run all**. Running on CPU *will* crash the
session (out of RAM) — cell 1 hard-stops if there is no GPU.

When it prints a URL, paste it here / set:
```bash
export DABWAYO_VIDEOGEN_URL='https://xxxx.trycloudflare.com'
export DABWAYO_VIDEOGEN_PROVIDER=remote
```

## 1. Require a GPU (hard stop on CPU) + show memory

In [ ]:
import torch, psutil, shutil
assert torch.cuda.is_available(), (
    'NO GPU. Runtime -> Change runtime type -> GPU (T4), then Restart and run all.')
print('GPU :', torch.cuda.get_device_name(0))
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9,1), 'GB')
print('RAM :', round(psutil.virtual_memory().total/1e9,1), 'GB total,',
      round(psutil.virtual_memory().available/1e9,1), 'GB free')
print('Disk:', round(shutil.disk_usage('/').free/1e9,1), 'GB free')

## 2. Install (minimal — does NOT touch torch)

In [ ]:
# Only the few packages Colab may lack. No torch/transformers pin => no heavy
# reinstall that can spike RAM or break CUDA.
%pip -q install diffusers accelerate imageio-ffmpeg fastapi "uvicorn[standard]" nest-asyncio
print('installed')

## 3. Load the small model (prints memory; shows any error)

In [ ]:
import gc, traceback, psutil
DTYPE = torch.float16            # T4 has no bfloat16
pipe = None
def _ram(tag): print(tag, '| RAM free', round(psutil.virtual_memory().available/1e9,1),
                     'GB | VRAM used', round(torch.cuda.memory_allocated()/1e9,2), 'GB')

def _load(variant):
    from diffusers import DiffusionPipeline
    kw = dict(torch_dtype=DTYPE, low_cpu_mem_usage=True)
    if variant: kw['variant'] = variant
    p = DiffusionPipeline.from_pretrained('damo-vilab/text-to-video-ms-1.7b', **kw)
    p.enable_model_cpu_offload()
    try: p.enable_vae_slicing()
    except Exception: pass
    return p

try:
    _ram('before load')
    print('loading weights (first run downloads ~3-7GB, be patient)...')
    try:
        pipe = _load('fp16')          # half-size weights if the repo has them
    except Exception as e:
        print('fp16 variant unavailable, retrying full weights:', repr(e)[:120])
        gc.collect(); torch.cuda.empty_cache()
        pipe = _load(None)
    _ram('after load ')
    print('READY: text-to-video-ms-1.7b')
except Exception:
    traceback.print_exc()
    print('\nLOAD FAILED — copy the LAST 2-3 red lines above and send them back.')

## 4. Generation API (`GET /health`, `POST /generate` -> video/mp4)

In [ ]:
import tempfile, traceback
import numpy as np
from fastapi import FastAPI, Request, Response, HTTPException
from diffusers.utils import export_to_video

def _to_uint8_frames(frames):
    """Normalise diffusers output (shape varies across versions) to a list of
    HxWx3 uint8 frames that export_to_video accepts."""
    if isinstance(frames, np.ndarray):
        if frames.ndim == 5:          # (batch, frames, H, W, C)
            frames = frames[0]
        frames = list(frames)          # -> list of (H, W, C)
    elif isinstance(frames, list) and frames and isinstance(frames[0], list):
        frames = frames[0]             # [[frame, ...]] -> [frame, ...]
    out = []
    for f in frames:
        f = np.asarray(f)
        if f.dtype != np.uint8:        # float in [0,1] -> 0..255
            f = (f.clip(0, 1) * 255).round().astype(np.uint8)
        out.append(f)
    return out

def run_generation(body):
    if pipe is None: raise HTTPException(503, 'model not loaded (see cell 3)')
    if body.get('mode','t2v') == 'i2v':
        raise HTTPException(400, 'this minimal server is t2v only')
    prompt = body.get('prompt',''); fps = float(body.get('fps',12))
    nf = int(body.get('num_frames',16)); steps = int(body.get('steps',20))
    seed = body.get('seed')
    gen = torch.Generator(device='cuda').manual_seed(int(seed)) if seed is not None else None
    raw = pipe(prompt, num_frames=nf, num_inference_steps=steps, generator=gen).frames
    frames = _to_uint8_frames(raw)     # tolerate every diffusers return shape
    path = tempfile.mktemp(suffix='.mp4'); export_to_video(frames, path, fps=fps)
    gc.collect(); torch.cuda.empty_cache()
    return open(path,'rb').read()

app = FastAPI()
@app.get('/health')
def health(): return {'ok': pipe is not None, 'backend': 'modelscope-1.7b',
                      'modes': ['t2v'], 'gpu': torch.cuda.get_device_name(0), 'build': 'patched-v2'}
@app.post('/generate')
async def generate(request: Request):
    body = await request.json()
    try:
        return Response(content=run_generation(body), media_type='video/mp4')
    except HTTPException:
        raise                                    # keep clean 4xx/503/507
    except torch.cuda.OutOfMemoryError:
        gc.collect(); torch.cuda.empty_cache()
        raise HTTPException(507, 'GPU OOM - lower num_frames')
    except Exception:
        tb = traceback.format_exc(); print(tb)   # full trace in the Colab log
        raise HTTPException(500, 'generation failed:\n' + tb[-1500:])
print('API defined')

## 5. Public tunnel + launch

In [ ]:
import nest_asyncio, threading, uvicorn, subprocess, re, time, os, urllib.request, stat, socket
nest_asyncio.apply()

# --- make this cell safe to re-run: stop the previous server + tunnel so the
#     freshly-defined `app` (cell 8) is what actually gets served. Without this
#     the old thread keeps port 8000 ("address already in use") and serves the
#     OLD code, so edits to cell 8 silently have no effect. ---
_old = globals().get('_uvicorn_server')
if _old is not None: _old.should_exit = True
_tun = globals().get('_tunnel_proc')
if _tun is not None:
    try: _tun.terminate()
    except Exception: pass
for _ in range(40):                       # wait until port 8000 is free
    s = socket.socket()
    try: s.bind(('0.0.0.0', 8000)); s.close(); break
    except OSError: s.close(); time.sleep(0.5)

_cfg = uvicorn.Config(app, host='0.0.0.0', port=8000, log_level='warning')
_uvicorn_server = uvicorn.Server(_cfg)
threading.Thread(target=_uvicorn_server.run, daemon=True).start()
time.sleep(3)

BIN='/usr/local/bin/cloudflared'
if not os.path.exists(BIN):
    urllib.request.urlretrieve('https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64', BIN)
    os.chmod(BIN, os.stat(BIN).st_mode | stat.S_IEXEC)
_tunnel_proc = subprocess.Popen([BIN,'tunnel','--url','http://localhost:8000','--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url=None
for line in _tunnel_proc.stdout:
    m=re.search(r'https://[\w.-]+\.trycloudflare\.com', line)
    if m: url=m.group(0); break
print('\n'+'='*60); print('DABWAYO_VIDEOGEN_URL =', url); print('='*60)
print(f"export DABWAYO_VIDEOGEN_URL='{url}'"); print('export DABWAYO_VIDEOGEN_PROVIDER=remote')
print('Keep this tab open.')

## 6. (Optional) Smoke test (tiny)

In [ ]:
import requests
print('health:', requests.get(url+'/health', timeout=30).json())
r=requests.post(url+'/generate', json={'prompt':'a corgi running on the beach',
    'num_frames':16,'fps':12,'steps':20}, timeout=900)
open('test.mp4','wb').write(r.content); print('wrote', len(r.content),'bytes')
from IPython.display import Video; Video('test.mp4', embed=True)